In [1]:
!pip -q install -U transformers accelerate bitsandbytes sentencepiece

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 104.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 44.5 MB/s eta 0:00:00


In [2]:
from huggingface_hub import notebook_login

notebook_login()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [ ]:
import os
import re
import json
import time
from datetime import datetime, UTC

import numpy as np
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

# REQUIRES MANUAL UPLOAD TO RUNTIME FROM LOCAl
JSONL_PATH = "/content/prep_rows.jsonl"
MODEL_NAME = "meta-llama/Llama-3.1-8B-Instruct"

# TRY WITH 10 TO TEST
# REPLACE WITH len(prep_rows) FOR FULL RUN
N_RUN = 10 
TARGET_REL = 2
RETURN_TOP_N = 5
MAX_NEW_TOKENS = 256
RETRIES = 2
SAVE_EVERY = 1
N_PRINT = 5

RUN_ROOT = "/content/trecdl_llama100_runs"
os.makedirs(RUN_ROOT, exist_ok=True)

RUN_NAME = "llama100_top5_" + datetime.now(UTC).strftime("%Y%m%d_%H%M%S")
RUN_DIR = os.path.join(RUN_ROOT, RUN_NAME)
os.makedirs(RUN_DIR, exist_ok=True)

def write_json(path, obj):
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, ensure_ascii=False, indent=2)

def write_jsonl(path, rows):
    with open(path, "w", encoding="utf-8") as f:
        for row in rows:
            f.write(json.dumps(row, ensure_ascii=False) + "\n")

def norm_ws(s):
    return re.sub(r"\s+", " ", (s or "")).strip()

def load_jsonl(path):
    rows = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            rows.append(json.loads(line))
    return rows

def build_candidates_block(docs):
    parts = []
    for i, d in enumerate(docs):
        parts.append(f"[{i}] {norm_ws(d)}")
    return "\n\n".join(parts)

def build_user_prompt(query, docs, return_top_n):
    return (
        "You are ranking evidence passages for a search query.\n\n"
        "Task:\n"
        f"- You are given exactly {len(docs)} candidate passages.\n"
        f"- Select the best {return_top_n} passages.\n"
        f"- Rank only those best {return_top_n} passages from best to worst.\n"
        "- Focus on which passages best answer the query.\n\n"
        f"Return ONLY strict JSON with exactly {return_top_n} unique indices:\n"
        '{"ranked_indices":[...]}\n\n'
        "QUERY:\n"
        f"{norm_ws(query)}\n\n"
        "CANDIDATE PASSAGES:\n"
        f"{build_candidates_block(docs)}\n\n"
        "Return JSON only."
    )

def extract_json_obj(text):
    s = (text or "").strip()
    if not s:
        return None
    try:
        return json.loads(s)
    except Exception:
        pass
    i = s.find("{")
    j = s.rfind("}")
    if i >= 0 and j > i:
        chunk = s[i:j+1]
        try:
            return json.loads(chunk)
        except Exception:
            return None
    return None

def clamp_indices(arr, k, expected_len):
    if not isinstance(arr, list):
        return None
    out = []
    seen = set()
    for x in arr:
        try:
            ix = int(x)
        except Exception:
            continue
        if 0 <= ix < k and ix not in seen:
            seen.add(ix)
            out.append(ix)
    return out if len(out) == expected_len else None

def precision_at_k(labels, ranked_indices, k, rel_threshold):
    top = ranked_indices[:k]
    if len(top) < k:
        return None
    good = sum(1 for ix in top if labels[ix] >= rel_threshold)
    return good / k

def recall_at_k(labels, ranked_indices, k, rel_threshold):
    top = ranked_indices[:k]
    if len(top) < k:
        return None
    total_good = sum(1 for x in labels if x >= rel_threshold)
    if total_good == 0:
        return 0.0
    good = sum(1 for ix in top if labels[ix] >= rel_threshold)
    return good / total_good

def hit_at_k(labels, ranked_indices, k, rel_threshold):
    top = ranked_indices[:k]
    if len(top) < k:
        return None
    return float(any(labels[ix] >= rel_threshold for ix in top))

def dcg_at_k(labels, ranked_indices, k):
    top = ranked_indices[:k]
    if len(top) < k:
        return None
    vals = []
    for rank_pos, ix in enumerate(top, start=1):
        rel = labels[ix]
        gain = (2 ** max(rel, 0) - 1) / np.log2(rank_pos + 1)
        vals.append(gain)
    return float(sum(vals))

def ndcg_at_k(labels, ranked_indices, k):
    actual = dcg_at_k(labels, ranked_indices, k)
    if actual is None:
        return None
    ideal_order = sorted(range(len(labels)), key=lambda i: labels[i], reverse=True)
    ideal = dcg_at_k(labels, ideal_order, k)
    if ideal == 0:
        return 0.0
    return actual / ideal

def generate_ranking_text(model, tokenizer, messages, max_new_tokens):
    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_dict=True,
        return_tensors="pt"
    )
    inputs = {k: v.to(model.device) for k, v in inputs.items()}

    input_token_count = int(inputs["input_ids"].shape[1])

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )

    generated = outputs[0][inputs["input_ids"].shape[1]:]
    text = tokenizer.decode(generated, skip_special_tokens=True)

    return text, input_token_count

def rank_single_example(model, tokenizer, ex, retries, max_new_tokens, return_top_n):
    query = ex["query"]
    docs = [c["passage_text"] for c in ex["candidates_100"]]

    user_prompt = build_user_prompt(query, docs, return_top_n)
    messages = [{"role": "user", "content": user_prompt}]

    last_text = ""
    last_obj = None
    input_token_count = None
    success = 0

    for attempt in range(retries + 1):
        text, input_token_count = generate_ranking_text(
            model=model,
            tokenizer=tokenizer,
            messages=messages,
            max_new_tokens=max_new_tokens
        )

        last_text = text
        last_obj = extract_json_obj(text)

        if last_obj and "ranked_indices" in last_obj:
            ranking = clamp_indices(last_obj["ranked_indices"], len(docs), return_top_n)
            if ranking is not None:
                success = 1
                return {
                    "success": success,
                    "attempts_used": attempt + 1,
                    "input_token_count": int(input_token_count),
                    "ranking": ranking,
                    "raw_text": last_text,
                    "parsed_obj": last_obj
                }

        time.sleep(0.25)

    return {
        "success": success,
        "attempts_used": retries + 1,
        "input_token_count": int(input_token_count) if input_token_count is not None else None,
        "ranking": None,
        "raw_text": last_text,
        "parsed_obj": last_obj
    }

if not os.path.exists(JSONL_PATH):
    raise FileNotFoundError(f"Uploaded JSONL file not found: {JSONL_PATH}")

prep_rows = load_jsonl(JSONL_PATH)

for ex in prep_rows:
    ex["trec_year"] = int(ex["trec_year"])
    ex["query_id"] = str(ex["query_id"])
    ex["query"] = norm_ws(ex["query"])
    ex["labels_100"] = [int(x) for x in ex["labels_100"]]
    ex["highly_rel_positions"] = [int(x) for x in ex["highly_rel_positions"]]
    ex["n_candidates"] = int(ex["n_candidates"])
    ex["n_highly_relevant"] = int(ex["n_highly_relevant"])

    clean_candidates = []
    for c in ex["candidates_100"]:
        clean_candidates.append({
            "doc_id": str(c["doc_id"]),
            "score": float(c["score"]),
            "passage_text": norm_ws(c["passage_text"]),
            "source_doc_id": str(c.get("source_doc_id", ""))
        })
    ex["candidates_100"] = clean_candidates

prep_rows = sorted(prep_rows, key=lambda x: (x["trec_year"], x["query_id"]))

print("Loaded prep_rows:", len(prep_rows))
print()

year_counts = (
    pd.DataFrame([{"trec_year": x["trec_year"]} for x in prep_rows])
    .groupby("trec_year")
    .size()
    .reset_index(name="n_queries")
)

print("Prepared queries by year")
print(year_counts.to_string(index=False))
print()

shape_df = pd.DataFrame([
    {
        "trec_year": x["trec_year"],
        "query_id": x["query_id"],
        "n_candidates": x["n_candidates"],
        "n_highly_relevant": x["n_highly_relevant"]
    }
    for x in prep_rows
])

print("Prepared dataset shape summary")
print(shape_df[["n_candidates", "n_highly_relevant"]].describe().to_string())
print()

bad_rows = []
for i, ex in enumerate(prep_rows):
    if len(ex["candidates_100"]) != 100:
        bad_rows.append((i, ex["trec_year"], ex["query_id"], "candidates_100_len", len(ex["candidates_100"])))
    if len(ex["labels_100"]) != 100:
        bad_rows.append((i, ex["trec_year"], ex["query_id"], "labels_100_len", len(ex["labels_100"])))
    if ex["n_candidates"] != 100:
        bad_rows.append((i, ex["trec_year"], ex["query_id"], "n_candidates_field", ex["n_candidates"]))
    if ex["n_highly_relevant"] < 5:
        bad_rows.append((i, ex["trec_year"], ex["query_id"], "n_highly_relevant_field", ex["n_highly_relevant"]))

print("Validation check")
if bad_rows:
    print("Found invalid rows:", len(bad_rows))
    for row in bad_rows[:20]:
        print(row)
    raise ValueError("Dataset validation failed")
else:
    print("All rows passed basic validation")
print()

for ex in prep_rows[:N_PRINT]:
    print("=" * 120)
    print("TREC YEAR:", ex["trec_year"])
    print("QUERY ID:", ex["query_id"])
    print("QUERY:", ex["query"])
    print("N CANDIDATES:", ex["n_candidates"])
    print("N HIGHLY RELEVANT:", ex["n_highly_relevant"])
    print("TOP HIGHLY RELEVANT POSITIONS:", ex["highly_rel_positions"][:10])
    print()

    preview_rows = []
    for j, c in enumerate(ex["candidates_100"][:5]):
        preview_rows.append({
            "cand_ix": j,
            "doc_id": c["doc_id"],
            "score": round(c["score"], 4),
            "relevance": ex["labels_100"][j],
            "passage_preview": c["passage_text"][:220]
        })

    print(pd.DataFrame(preview_rows).to_string(index=False))
    print()

print("Loading tokenizer and model")
print("Model name:", MODEL_NAME)
print()

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4"
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto"
)

run_rows = prep_rows[:N_RUN]

meta = {
    "run_name": RUN_NAME,
    "created_utc": datetime.now(UTC).strftime("%Y-%m-%d %H:%M:%S"),
    "jsonl_path": JSONL_PATH,
    "n_loaded_queries": int(len(prep_rows)),
    "n_run": int(len(run_rows)),
    "target_rel": int(TARGET_REL),
    "return_top_n": int(RETURN_TOP_N),
    "max_new_tokens": int(MAX_NEW_TOKENS),
    "retries": int(RETRIES),
    "model_name": getattr(model.config, "_name_or_path", MODEL_NAME)
}
write_json(os.path.join(RUN_DIR, "meta.json"), meta)

print("Run configuration")
print(json.dumps(meta, indent=2))
print()

results_rows = []
raw_rows = []

for idx, ex in enumerate(run_rows):
    t0 = time.time()

    out = rank_single_example(
        model=model,
        tokenizer=tokenizer,
        ex=ex,
        retries=RETRIES,
        max_new_tokens=MAX_NEW_TOKENS,
        return_top_n=RETURN_TOP_N
    )

    elapsed = time.time() - t0
    labels = ex["labels_100"]

    if out["ranking"] is not None:
        ranking = out["ranking"]
        top3 = ranking[:3]
        top5 = ranking[:5]

        p3 = precision_at_k(labels, ranking, 3, TARGET_REL)
        r3 = recall_at_k(labels, ranking, 3, TARGET_REL)
        hit3 = hit_at_k(labels, ranking, 3, TARGET_REL)
        ndcg3 = ndcg_at_k(labels, ranking, 3)

        p5 = precision_at_k(labels, ranking, 5, TARGET_REL)
        r5 = recall_at_k(labels, ranking, 5, TARGET_REL)
        hit5 = hit_at_k(labels, ranking, 5, TARGET_REL)
        ndcg5 = ndcg_at_k(labels, ranking, 5)
    else:
        ranking = None
        top3 = None
        top5 = None
        p3 = None
        r3 = None
        hit3 = None
        ndcg3 = None
        p5 = None
        r5 = None
        hit5 = None
        ndcg5 = None

    result_row = {
        "idx": int(idx),
        "trec_year": int(ex["trec_year"]),
        "query_id": ex["query_id"],
        "success": int(out["success"]),
        "attempts_used": int(out["attempts_used"]),
        "elapsed_sec": float(elapsed),
        "input_token_count": out["input_token_count"],
        "n_candidates": int(len(ex["candidates_100"])),
        "n_highly_relevant": int(sum(1 for x in labels if x >= TARGET_REL)),
        "top3": top3,
        "top5": top5,
        "p_at_3_rel2": p3,
        "r_at_3_rel2": r3,
        "hit_at_3_rel2": hit3,
        "ndcg_at_3": ndcg3,
        "p_at_5_rel2": p5,
        "r_at_5_rel2": r5,
        "hit_at_5_rel2": hit5,
        "ndcg_at_5": ndcg5
    }
    results_rows.append(result_row)

    raw_row = {
        "idx": int(idx),
        "trec_year": int(ex["trec_year"]),
        "query_id": ex["query_id"],
        "query": ex["query"],
        "success": int(out["success"]),
        "attempts_used": int(out["attempts_used"]),
        "input_token_count": out["input_token_count"],
        "raw_text": out["raw_text"],
        "parsed_obj": out["parsed_obj"]
    }
    raw_rows.append(raw_row)

    if (idx + 1) % SAVE_EVERY == 0:
        write_jsonl(os.path.join(RUN_DIR, "results_rows.jsonl"), results_rows)
        write_jsonl(os.path.join(RUN_DIR, "raw_rows.jsonl"), raw_rows)

    done = idx + 1
    success_rate = np.mean([r["success"] for r in results_rows])

    valid_ndcg3 = [r["ndcg_at_3"] for r in results_rows if r["ndcg_at_3"] is not None]
    valid_ndcg5 = [r["ndcg_at_5"] for r in results_rows if r["ndcg_at_5"] is not None]

    mean_ndcg3 = float(np.mean(valid_ndcg3)) if valid_ndcg3 else None
    mean_ndcg5 = float(np.mean(valid_ndcg5)) if valid_ndcg5 else None

    print(
        f"{done}/{len(run_rows)} done | "
        f"success_rate={success_rate:.3f} | "
        f"mean_ndcg3={mean_ndcg3 if mean_ndcg3 is not None else 'NA'} | "
        f"mean_ndcg5={mean_ndcg5 if mean_ndcg5 is not None else 'NA'} | "
        f"last_tokens={out['input_token_count']} | "
        f"last_time={elapsed:.2f}s"
    )

write_jsonl(os.path.join(RUN_DIR, "results_rows.jsonl"), results_rows)
write_jsonl(os.path.join(RUN_DIR, "raw_rows.jsonl"), raw_rows)

results_df = pd.DataFrame(results_rows)

summary = {
    "n_queries": int(len(results_df)),
    "success_rate": float(results_df["success"].mean()) if len(results_df) else 0.0,
    "mean_input_token_count": float(results_df["input_token_count"].dropna().mean()) if len(results_df["input_token_count"].dropna()) else None,
    "mean_elapsed_sec": float(results_df["elapsed_sec"].mean()) if len(results_df) else None,
    "metrics": {
        "p_at_3_rel2": float(results_df["p_at_3_rel2"].dropna().mean()) if len(results_df["p_at_3_rel2"].dropna()) else None,
        "r_at_3_rel2": float(results_df["r_at_3_rel2"].dropna().mean()) if len(results_df["r_at_3_rel2"].dropna()) else None,
        "hit_at_3_rel2": float(results_df["hit_at_3_rel2"].dropna().mean()) if len(results_df["hit_at_3_rel2"].dropna()) else None,
        "ndcg_at_3": float(results_df["ndcg_at_3"].dropna().mean()) if len(results_df["ndcg_at_3"].dropna()) else None,
        "p_at_5_rel2": float(results_df["p_at_5_rel2"].dropna().mean()) if len(results_df["p_at_5_rel2"].dropna()) else None,
        "r_at_5_rel2": float(results_df["r_at_5_rel2"].dropna().mean()) if len(results_df["r_at_5_rel2"].dropna()) else None,
        "hit_at_5_rel2": float(results_df["hit_at_5_rel2"].dropna().mean()) if len(results_df["hit_at_5_rel2"].dropna()) else None,
        "ndcg_at_5": float(results_df["ndcg_at_5"].dropna().mean()) if len(results_df["ndcg_at_5"].dropna()) else None
    }
}

write_json(os.path.join(RUN_DIR, "summary.json"), summary)

print()
print("Run finished")
print("Run directory:", RUN_DIR)
display(results_df)
print(json.dumps(summary, indent=2))

Loaded prep_rows: 125

Prepared queries by year
 trec_year  n_queries
      2021         40
      2022         42
      2023         43

Prepared dataset shape summary
       n_candidates  n_highly_relevant
count         125.0         125.000000
mean          100.0          14.936000
std             0.0          10.383566
min           100.0           5.000000
25%           100.0           8.000000
50%           100.0          10.000000
75%           100.0          20.000000
max           100.0          52.000000

Validation check
All rows passed basic validation

TREC YEAR: 2021
QUERY ID: 1040198
QUERY: who is the final arbiter of florida law in instances where there is no federal authority?
N CANDIDATES: 100
N HIGHLY RELEVANT: 8
TOP HIGHLY RELEVANT POSITIONS: [0, 1, 2, 3, 7, 9, 16, 62]

 cand_ix                       doc_id   score  relevance                                                                                                                                                

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Run configuration
{
  "run_name": "llama100_top5_20260412_234140",
  "created_utc": "2026-04-12 23:41:48",
  "jsonl_path": "/content/prep_rows.jsonl",
  "n_loaded_queries": 125,
  "n_run": 125,
  "target_rel": 2,
  "return_top_n": 5,
  "max_new_tokens": 256,
  "retries": 2,
  "model_name": "meta-llama/Llama-3.1-8B-Instruct"
}

1/125 done | success_rate=1.000 | mean_ndcg3=0.49720136497561807 | mean_ndcg5=0.38846558542095355 | last_tokens=7023 | last_time=3.92s
2/125 done | success_rate=1.000 | mean_ndcg3=0.6145210464813071 | mean_ndcg5=0.5317273381618441 | last_tokens=7583 | last_time=4.14s
3/125 done | success_rate=1.000 | mean_ndcg3=0.743014030987538 | mean_ndcg5=0.5953937496562133 | last_tokens=6658 | last_time=3.81s
4/125 done | success_rate=1.000 | mean_ndcg3=0.7290474022368608 | mean_ndcg5=0.6400186113632239 | last_tokens=7051 | last_time=3.98s
5/125 done | success_rate=1.000 | mean_ndcg3=0.6677732500929209 | mean_ndcg5=0.618596068710042 | last_tokens=6532 | last_time=3.80s
6/125 

,idx,trec_year,query_id,success,attempts_used,elapsed_sec,input_token_count,n_candidates,n_highly_relevant,top3,top5,p_at_3_rel2,r_at_3_rel2,hit_at_3_rel2,ndcg_at_3,p_at_5_rel2,r_at_5_rel2,hit_at_5_rel2,ndcg_at_5
0,0,2021,1040198,1,1,3.921111,7023,100,8,"[0, 3, 10]","[0, 3, 10, 49, 64]",0.666667,0.250000,1.0,0.497201,0.4,0.250000,1.0,0.388466
1,1,2021,1104300,1,1,4.137795,7583,100,52,"[0, 1, 2]","[0, 1, 2, 5, 11]",1.000000,0.057692,1.0,0.731841,0.8,0.076923,1.0,0.674989
2,2,2021,1104447,1,1,3.814578,6658,100,39,"[4, 5, 6]","[4, 5, 6, 20, 72]",1.000000,0.076923,1.0,1.000000,0.6,0.076923,1.0,0.722727
3,3,2021,1107821,1,1,3.977853,7051,100,12,"[0, 1, 2]","[0, 1, 2, 11, 28]",0.666667,0.166667,1.0,0.687148,0.8,0.333333,1.0,0.773893
4,4,2021,1109840,1,1,3.802130,6532,100,13,"[0, 4, 1]","[0, 4, 1, 59, 64]",0.666667,0.153846,1.0,0.422677,0.8,0.307692,1.0,0.532906
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
120,120,2023,3100289,1,1,4.375430,7816,100,6,"[0, 28, 18]","[0, 28, 18, 5, 72]",0.333333,0.166667,1.0,0.469279,0.2,0.166667,1.0,0.339160
121,121,2023,3100569,1,1,4.098709,7259,100,5,"[0, 2, 3]","[0, 2, 3, 4, 25]",0.000000,0.000000,0.0,0.087557,0.0,0.000000,0.0,0.126778
122,122,2023,3100825,1,1,3.891260,6583,100,6,"[0, 4, 14]","[0, 4, 14, 31, 60]",0.333333,0.166667,1.0,0.374295,0.4,0.333333,1.0,0.401718
123,123,2023,3100909,1,1,3.716528,6207,100,9,"[0, 1, 2]","[0, 1, 2, 4, 41]",0.333333,0.111111,1.0,0.396642,0.2,0.111111,1.0,0.307530


{
  "n_queries": 125,
  "success_rate": 0.984,
  "mean_input_token_count": 7101.16,
  "mean_elapsed_sec": 4.208381452560425,
  "metrics": {
    "p_at_3_rel2": 0.4688346883468835,
    "r_at_3_rel2": 0.11505564727242819,
    "hit_at_3_rel2": 0.7479674796747967,
    "ndcg_at_3": 0.4400113003354721,
    "p_at_5_rel2": 0.4634146341463415,
    "r_at_5_rel2": 0.1849795627601497,
    "hit_at_5_rel2": 0.8861788617886179,
    "ndcg_at_5": 0.44727609125567386
  }
}
